# Hybrid Robotic Vision: Notebook 5 — Bitrate Accounting and Hybrid System Summary

This notebook combines the outputs of:

- **Notebook 1**: base-video encoding metrics
- **Notebook 3**: ROI still-compression metrics
- **Notebook 4**: semantic-gain metrics

Its purpose is to compute the first **system-level hybrid results** for the paper.

## Main outputs of this notebook

### System-level bitrate metrics
- `video_bitrate_mbps`
- `roi_bitrate_mbps`
- `hybrid_bitrate_mbps`
- `roi_bitrate_share`

### Coverage / selection metrics
- `num_raw_detections`
- `num_selected_rois`
- `roi_selection_ratio`
- `frame_roi_coverage`
- `small_object_roi_ratio`
- `roi_rate_hz`
- `mean_roi_bytes`
- `total_roi_bytes`

### Semantic-gain metrics
- `mean_video_conf`
- `mean_still_conf`
- `mean_conf_gain`
- `median_conf_gain`
- `positive_conf_gain_rate`
- `pred_change_rate`
- `small_object_mean_conf_gain`
- `mean_entropy_gain`

### Hybrid utility metrics
- `conf_gain_per_kb`
- `positive_gain_events_per_mb`
- `mean_conf_gain_per_selected_roi`

## Inputs expected

This notebook reads:
- Notebook 1 `video_only_metrics.json`
- Notebook 2 `detections.csv`, `roi_candidates.csv`, `roi_manifest.csv`, `run_summary.json`
- Notebook 3 `roi_still_manifest.csv`, `run_summary.json`
- Notebook 4 `semantic_gain_summary.json`, `classification_results.csv`, `run_summary.json`

## Outputs produced

```text
runs/
  hybrid_summary/
    <sequence_name>/
      <regime>/
        hybrid_system_summary.json
        hybrid_system_summary.csv
        coverage_selection_table.csv
        semantic_gain_table.csv
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Define paths and select the experiment run

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/hybrid_robotic_vision")
RUNS = ROOT / "runs"

SEQUENCE_NAME = "M0101"   # <-- EDIT THIS
REGIME = "low"            # <-- choose: "low" or "moderate"

# Notebook 1 outputs
if REGIME == "low":
    VIDEO_ONLY_DIR = RUNS / "uavdt_low" / "video_only" / SEQUENCE_NAME
elif REGIME == "moderate":
    VIDEO_ONLY_DIR = RUNS / "uavdt_moderate" / "video_only" / SEQUENCE_NAME
else:
    raise ValueError("REGIME must be 'low' or 'moderate'")

VIDEO_ONLY_METRICS_JSON = VIDEO_ONLY_DIR / "video_only_metrics.json"

# Notebook 2 outputs
N2_BASE = RUNS / "yolo_roi_manifest" / SEQUENCE_NAME / REGIME
DETECTIONS_CSV = N2_BASE / "detections" / "detections.csv"
ROI_CANDIDATES_CSV = N2_BASE / "roi_candidates" / "roi_candidates.csv"
ROI_MANIFEST_CSV = N2_BASE / "manifests" / "roi_manifest.csv"
N2_SUMMARY_JSON = N2_BASE / "manifests" / "run_summary.json"

# Notebook 3 outputs
N3_BASE = RUNS / "roi_still_compression_jpeg" / SEQUENCE_NAME / REGIME / "manifests"
ROI_STILL_MANIFEST_CSV = N3_BASE / "roi_still_manifest.csv"
N3_SUMMARY_JSON = N3_BASE / "run_summary.json"

# Notebook 4 outputs
N4_BASE = RUNS / "classification_semantic_gain" / SEQUENCE_NAME / REGIME / "manifests"
CLASSIFICATION_RESULTS_CSV = N4_BASE / "classification_results.csv"
SEMANTIC_GAIN_SUMMARY_JSON = N4_BASE / "semantic_gain_summary.json"
N4_SUMMARY_JSON = N4_BASE / "run_summary.json"

# Output dir
OUT_DIR = RUNS / "hybrid_summary" / SEQUENCE_NAME / REGIME
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook 1 metrics:", VIDEO_ONLY_METRICS_JSON)
print("Notebook 2 ROI manifest:", ROI_MANIFEST_CSV)
print("Notebook 3 still manifest:", ROI_STILL_MANIFEST_CSV)
print("Notebook 4 semantic summary:", SEMANTIC_GAIN_SUMMARY_JSON)
print("Output dir:", OUT_DIR)

## 3. Imports

In [ ]:
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path

## 4. Load all available inputs

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    with open(path, "r") as f:
        return json.load(f)

def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

video_metrics = load_json(VIDEO_ONLY_METRICS_JSON)
n2_summary = load_json(N2_SUMMARY_JSON)
n3_summary = load_json(N3_SUMMARY_JSON)
semantic_summary = load_json(SEMANTIC_GAIN_SUMMARY_JSON)
n4_summary = load_json(N4_SUMMARY_JSON)

detections_df = load_csv(DETECTIONS_CSV)
roi_candidates_df = load_csv(ROI_CANDIDATES_CSV)
roi_manifest_df = load_csv(ROI_MANIFEST_CSV)
roi_still_df = load_csv(ROI_STILL_MANIFEST_CSV)
classification_results_df = load_csv(CLASSIFICATION_RESULTS_CSV)

print("Loaded all required inputs.")
print("detections_df:", detections_df.shape)
print("roi_candidates_df:", roi_candidates_df.shape)
print("roi_manifest_df:", roi_manifest_df.shape)
print("roi_still_df:", roi_still_df.shape)
print("classification_results_df:", classification_results_df.shape)

## 5. Compute coverage metrics

In [ ]:
num_raw_detections = int(len(detections_df))
num_selected_rois = int(len(roi_manifest_df))

roi_selection_ratio = (
    num_selected_rois / num_raw_detections if num_raw_detections > 0 else None
)

processed_frames = (
    int(detections_df["enc_frame_idx"].nunique()) if "enc_frame_idx" in detections_df.columns and len(detections_df) > 0 else 0
)

frames_with_rois = (
    int(roi_manifest_df["enc_frame_idx"].nunique()) if "enc_frame_idx" in roi_manifest_df.columns and len(roi_manifest_df) > 0 else 0
)

frame_roi_coverage = (
    frames_with_rois / processed_frames if processed_frames > 0 else None
)

if "area_frac" in roi_manifest_df.columns and len(roi_manifest_df) > 0:
    small_object_mask = roi_manifest_df["area_frac"] < 0.015
    small_object_roi_ratio = float(small_object_mask.mean())
    num_small_object_rois = int(small_object_mask.sum())
else:
    small_object_roi_ratio = None
    num_small_object_rois = 0

coverage_metrics = {
    "num_raw_detections": num_raw_detections,
    "num_selected_rois": num_selected_rois,
    "roi_selection_ratio": roi_selection_ratio,
    "processed_frames": processed_frames,
    "frames_with_rois": frames_with_rois,
    "frame_roi_coverage": frame_roi_coverage,
    "small_object_roi_ratio": small_object_roi_ratio,
    "num_small_object_rois": num_small_object_rois,
}

coverage_metrics

## 6. Compute selection and ROI-cost metrics

In [ ]:
def safe_mean(series):
    series = pd.Series(series).dropna()
    return float(series.mean()) if len(series) > 0 else None

def safe_median(series):
    series = pd.Series(series).dropna()
    return float(series.median()) if len(series) > 0 else None

duration_s = float(video_metrics.get("duration_s")) if video_metrics.get("duration_s") is not None else None

# Use Notebook 3's per-ROI still byte sizes when available
if "still_size_bytes" in roi_still_df.columns:
    valid_still_sizes = roi_still_df["still_size_bytes"].dropna()
else:
    valid_still_sizes = pd.Series(dtype=float)

total_roi_bytes = int(valid_still_sizes.sum()) if len(valid_still_sizes) > 0 else 0
mean_roi_bytes = safe_mean(valid_still_sizes)
median_roi_bytes = safe_median(valid_still_sizes)

roi_rate_hz = (
    num_selected_rois / duration_s if duration_s is not None and duration_s > 0 else None
)

mean_rois_per_processed_frame = (
    num_selected_rois / processed_frames if processed_frames > 0 else None
)

roi_bitrate_mbps = (
    (8.0 * total_roi_bytes) / duration_s / 1e6 if duration_s is not None and duration_s > 0 else None
)

# Trigger reason distribution
if "roi_reason" in roi_manifest_df.columns and len(roi_manifest_df) > 0:
    roi_reason_counts = roi_manifest_df["roi_reason"].value_counts().to_dict()
    roi_reason_distribution = {str(k): float(v) / num_selected_rois for k, v in roi_reason_counts.items()}
else:
    roi_reason_counts = {}
    roi_reason_distribution = {}

selection_metrics = {
    "duration_s": duration_s,
    "roi_rate_hz": roi_rate_hz,
    "mean_rois_per_processed_frame": mean_rois_per_processed_frame,
    "total_roi_bytes": total_roi_bytes,
    "mean_roi_bytes": mean_roi_bytes,
    "median_roi_bytes": median_roi_bytes,
    "roi_bitrate_mbps": roi_bitrate_mbps,
    "roi_reason_counts": roi_reason_counts,
    "roi_reason_distribution": roi_reason_distribution,
}

selection_metrics

## 7. Compute base-video and hybrid bitrate metrics

In [ ]:
video_bitrate_mbps = (
    float(video_metrics.get("achieved_bitrate_mbps"))
    if video_metrics.get("achieved_bitrate_mbps") is not None
    else None
)

target_video_bitrate_mbps = (
    float(video_metrics.get("target_bitrate_mbps"))
    if video_metrics.get("target_bitrate_mbps") is not None
    else None
)

hybrid_bitrate_mbps = (
    video_bitrate_mbps + roi_bitrate_mbps
    if video_bitrate_mbps is not None and roi_bitrate_mbps is not None
    else None
)

roi_bitrate_share = (
    roi_bitrate_mbps / hybrid_bitrate_mbps
    if hybrid_bitrate_mbps is not None and hybrid_bitrate_mbps > 0
    else None
)

bitrate_metrics = {
    "target_video_bitrate_mbps": target_video_bitrate_mbps,
    "video_bitrate_mbps": video_bitrate_mbps,
    "roi_bitrate_mbps": roi_bitrate_mbps,
    "hybrid_bitrate_mbps": hybrid_bitrate_mbps,
    "roi_bitrate_share": roi_bitrate_share,
}

bitrate_metrics

## 8. Read semantic-gain metrics from Notebook 4

In [ ]:
semantic_metrics = {
    "num_classified_rois": int(semantic_summary.get("num_classified_rois", 0)),
    "mean_video_conf": semantic_summary.get("mean_video_conf"),
    "mean_still_conf": semantic_summary.get("mean_still_conf"),
    "mean_conf_gain": semantic_summary.get("mean_conf_gain"),
    "median_conf_gain": semantic_summary.get("median_conf_gain"),
    "positive_conf_gain_rate": semantic_summary.get("positive_conf_gain_rate"),
    "pred_change_rate": semantic_summary.get("pred_change_rate"),
    "mean_video_entropy": semantic_summary.get("mean_video_entropy"),
    "mean_still_entropy": semantic_summary.get("mean_still_entropy"),
    "mean_entropy_gain": semantic_summary.get("mean_entropy_gain"),
    "small_object_mean_conf_gain": semantic_summary.get("small_object_mean_conf_gain"),
    "small_object_count": semantic_summary.get("small_object_count"),
}

semantic_metrics

## 9. Compute system-level utility metrics

In [ ]:
mean_conf_gain = semantic_metrics.get("mean_conf_gain")
num_classified_rois = semantic_metrics.get("num_classified_rois", 0)

# Uses total confidence gain across classified ROIs, normalized by ROI bytes
if len(classification_results_df) > 0 and "conf_gain" in classification_results_df.columns:
    total_conf_gain = float(classification_results_df["conf_gain"].dropna().sum())
    total_positive_gain_events = int(classification_results_df["positive_conf_gain"].sum()) if "positive_conf_gain" in classification_results_df.columns else None
else:
    total_conf_gain = None
    total_positive_gain_events = None

conf_gain_per_kb = (
    total_conf_gain / (total_roi_bytes / 1024.0)
    if total_conf_gain is not None and total_roi_bytes > 0
    else None
)

positive_gain_events_per_mb = (
    total_positive_gain_events / (total_roi_bytes / (1024.0 * 1024.0))
    if total_positive_gain_events is not None and total_roi_bytes > 0
    else None
)

mean_conf_gain_per_selected_roi = (
    total_conf_gain / num_selected_rois
    if total_conf_gain is not None and num_selected_rois > 0
    else None
)

system_utility_metrics = {
    "total_conf_gain": total_conf_gain,
    "total_positive_gain_events": total_positive_gain_events,
    "conf_gain_per_kb": conf_gain_per_kb,
    "positive_gain_events_per_mb": positive_gain_events_per_mb,
    "mean_conf_gain_per_selected_roi": mean_conf_gain_per_selected_roi,
}

system_utility_metrics

## 10. Build the full hybrid system summary

In [ ]:
hybrid_summary = {
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,

    **coverage_metrics,
    **selection_metrics,
    **bitrate_metrics,
    **semantic_metrics,
    **system_utility_metrics,

    "source_video_metrics_json": str(VIDEO_ONLY_METRICS_JSON),
    "source_n2_summary_json": str(N2_SUMMARY_JSON),
    "source_n3_summary_json": str(N3_SUMMARY_JSON),
    "source_n4_summary_json": str(N4_SUMMARY_JSON),
    "source_roi_manifest_csv": str(ROI_MANIFEST_CSV),
    "source_roi_still_manifest_csv": str(ROI_STILL_MANIFEST_CSV),
    "source_classification_results_csv": str(CLASSIFICATION_RESULTS_CSV),
}

print(json.dumps(hybrid_summary, indent=2))

## 11. Save the main hybrid system summary

In [ ]:
hybrid_summary_json = OUT_DIR / "hybrid_system_summary.json"
with open(hybrid_summary_json, "w") as f:
    json.dump(hybrid_summary, f, indent=2)

hybrid_summary_csv = OUT_DIR / "hybrid_system_summary.csv"
pd.DataFrame([hybrid_summary]).to_csv(hybrid_summary_csv, index=False)

print("Saved:")
print(hybrid_summary_json)
print(hybrid_summary_csv)

## 12. Save paper-oriented tables

In [ ]:
coverage_selection_table = pd.DataFrame([{
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "num_raw_detections": num_raw_detections,
    "num_selected_rois": num_selected_rois,
    "roi_selection_ratio": roi_selection_ratio,
    "frame_roi_coverage": frame_roi_coverage,
    "small_object_roi_ratio": small_object_roi_ratio,
    "roi_rate_hz": roi_rate_hz,
    "mean_roi_bytes": mean_roi_bytes,
    "total_roi_bytes": total_roi_bytes,
    "roi_bitrate_mbps": roi_bitrate_mbps,
    "roi_bitrate_share": roi_bitrate_share,
}])

semantic_gain_table = pd.DataFrame([{
    "sequence_name": SEQUENCE_NAME,
    "regime": REGIME,
    "num_classified_rois": semantic_metrics["num_classified_rois"],
    "mean_video_conf": semantic_metrics["mean_video_conf"],
    "mean_still_conf": semantic_metrics["mean_still_conf"],
    "mean_conf_gain": semantic_metrics["mean_conf_gain"],
    "median_conf_gain": semantic_metrics["median_conf_gain"],
    "positive_conf_gain_rate": semantic_metrics["positive_conf_gain_rate"],
    "pred_change_rate": semantic_metrics["pred_change_rate"],
    "small_object_mean_conf_gain": semantic_metrics["small_object_mean_conf_gain"],
    "mean_entropy_gain": semantic_metrics["mean_entropy_gain"],
    "conf_gain_per_kb": conf_gain_per_kb,
}])

coverage_selection_csv = OUT_DIR / "coverage_selection_table.csv"
semantic_gain_csv = OUT_DIR / "semantic_gain_table.csv"

coverage_selection_table.to_csv(coverage_selection_csv, index=False)
semantic_gain_table.to_csv(semantic_gain_csv, index=False)

print("Saved:")
print(coverage_selection_csv)
print(semantic_gain_csv)

display(coverage_selection_table)
display(semantic_gain_table)

## 13. Optional detailed diagnostics

In [ ]:
# Class-wise diagnostic table from Notebook 4 outputs
if len(classification_results_df) > 0 and "cls_name" in classification_results_df.columns:
    classwise_diag = classification_results_df.groupby("cls_name").agg(
        num_rois=("cls_name", "size"),
        mean_video_conf=("video_pred_conf", "mean"),
        mean_still_conf=("still_pred_conf", "mean"),
        mean_conf_gain=("conf_gain", "mean"),
        positive_conf_gain_rate=("positive_conf_gain", "mean"),
        pred_change_rate=("pred_changed", "mean"),
    ).reset_index().sort_values("num_rois", ascending=False)

    classwise_diag_csv = OUT_DIR / "classwise_semantic_gain_diagnostics.csv"
    classwise_diag.to_csv(classwise_diag_csv, index=False)

    print("Saved:")
    print(classwise_diag_csv)
    display(classwise_diag.head(20))
else:
    print("No class-wise diagnostics available.")

## 14. Human-readable summary block

In [ ]:
print("=== HYBRID SYSTEM SUMMARY ===")
print(f"Sequence: {SEQUENCE_NAME}")
print(f"Regime: {REGIME}")
print()
print(f"Video bitrate (Mbps): {video_bitrate_mbps}")
print(f"ROI bitrate (Mbps): {roi_bitrate_mbps}")
print(f"Hybrid bitrate (Mbps): {hybrid_bitrate_mbps}")
print(f"ROI bitrate share: {roi_bitrate_share}")
print()
print(f"Raw detections: {num_raw_detections}")
print(f"Selected ROIs: {num_selected_rois}")
print(f"ROI selection ratio: {roi_selection_ratio}")
print(f"Frame ROI coverage: {frame_roi_coverage}")
print(f"ROI rate (Hz): {roi_rate_hz}")
print()
print(f"Mean ROI bytes: {mean_roi_bytes}")
print(f"Total ROI bytes: {total_roi_bytes}")
print()
print(f"Num classified ROIs: {semantic_metrics['num_classified_rois']}")
print(f"Mean video confidence: {semantic_metrics['mean_video_conf']}")
print(f"Mean still confidence: {semantic_metrics['mean_still_conf']}")
print(f"Mean confidence gain: {semantic_metrics['mean_conf_gain']}")
print(f"Positive confidence gain rate: {semantic_metrics['positive_conf_gain_rate']}")
print(f"Prediction change rate: {semantic_metrics['pred_change_rate']}")
print(f"Small-object mean confidence gain: {semantic_metrics['small_object_mean_conf_gain']}")
print(f"Mean entropy gain: {semantic_metrics['mean_entropy_gain']}")
print()
print(f"Confidence gain per KB: {conf_gain_per_kb}")
print(f"Positive gain events per MB: {positive_gain_events_per_mb}")

## 15. What this notebook completed

At this point you now have the first **system-level hybrid summary** for one sequence and one bitrate regime.

This notebook combined:

- base-video bitrate from Notebook 1
- ROI still bitrate from Notebook 3
- semantic-gain metrics from Notebook 4

and produced:
- a full hybrid summary JSON / CSV
- a coverage and selection table
- a semantic-gain table
- utility-per-byte metrics

## What to do next

1. Run the same notebook for:
   - the second bitrate regime
   - more sequences

2. Then build Notebook 6 for:
   - multi-sequence aggregation
   - plots for the paper
   - final result tables